# NYC Taxi Ride Duration: Does Weather Help?
**Negar Kamvij | CS672 Project 2 | Due November 13, 2026**

**Status: project starter. Models have not been trained and no results are claimed.**

## Project objective
Build and compare TensorFlow models that predict NYC Yellow Taxi trip duration using taxi and weather data.

## My research question
Does adding weather information improve predictions for later taxi trips, especially during rainy or snowy periods?

## My expectation before training
I expect rain and snow to be associated with longer rides for similar routes and pickup times. This is a hypothesis to test, not a result. Weather may also change who travels and which routes they take. Trip duration is not a direct measurement of traffic.

## Prediction moment
The prediction is made when the ride starts and the destination is known. Candidate inputs are the pickup zone, destination zone, time, and weather information available then. Actual drop-off time, final fare, tips, tolls, and completed-trip distance are excluded from inputs. Pickup and destination zones are areas, not exact GPS points. An estimated route distance may be added later only if its source is documented and it is available before departure.


## 1. Assignment checklist
This checklist records the work required by the assignment. The notebook currently prepares the audit; remaining items are planned.

| Requirement | Planned evidence | Status |
| --- | --- | --- |
| Combine NYC taxi and weather data | Time-zone checks, weather definitions, merge coverage report | Pending weather selection |
| Prepare numeric inputs and scale features | Training-only preprocessing; documented categorical encoding | Planned |
| Time-sensitive 80/20 train/validation split | Chronological boundary and checks for overlapping rides | Planned |
| TensorFlow linear regression | Sequential model with no hidden layers | Planned |
| TensorFlow MLP | One hidden layer for a clear architecture comparison | Planned |
| TensorFlow DNN | At least two hidden layers | Planned |
| MSE and MAE losses | Experiment table | Planned |
| SGD, Adam, RMSProp; multiple learning rates | Reproducible experiment configurations | Planned |
| 100 epochs per fit; batch size 32 | Training logs | Planned |
| Training and validation loss after every fit | Saved curve for every run | Planned |
| Best model and parameters | Common validation metrics, exact configuration | Planned |
| `model.predict()` and error review | Predictions, MAE in minutes, RMSE, residuals | Planned |
| PyTorch extra credit | DNN with at least two hidden layers and fair framework comparison | Planned |

Additional investigation: weather versus no-weather inputs, rainy/dry error comparison, and examples of large prediction errors. If snow is rare or unavailable, report that limitation.

Source: CS672 Project 2 assignment supplied with this project. The PDF is not included in the public repository.


## 2. Setup
Open this notebook in Colab or another Jupyter-compatible environment. Run cells in order. The initial audit uses pandas, NumPy, and PyArrow. If needed, run `%pip install pandas numpy pyarrow` in a separate cell, then restart the runtime.

Raw files and trained models stay outside Git. The download cell uses the official TLC host and downloads only if the monthly file is missing. Downloading and auditing the full January file needs disk space and memory; it does not train a model.

January 2020 is the starting candidate because the assignment supplies a January weather example. This is a project choice, not a statement that January is mandatory. The audit and weather coverage will determine whether a longer period is needed for the weather question.


In [ ]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from urllib.request import urlretrieve
import json
import platform
import numpy as np
import pandas as pd

SEED = 672
TAXI_MONTH = "2020-01"
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)
print("Python:", platform.python_version())
for package in ["pandas", "numpy", "pyarrow", "tensorflow", "torch"]:
    try:
        print(package, version(package))
    except PackageNotFoundError:
        print(package, "not installed; only needed when its section is implemented")
print("Audit month:", TAXI_MONTH)


## 3. Download the original taxi data
The source file stays unchanged. The audit reads selected columns from every row. Selecting columns reduces memory use and does not remove trips.


In [ ]:
def download_taxi_month(month: str, data_dir: Path) -> Path:
    """Return a local TLC Parquet file, downloading it if missing.

    Args:
        month (str): Calendar month in YYYY-MM format.
        data_dir (Path): Directory for the original monthly file.
    Returns:
        Path: Path to the source Parquet file.
    Raises:
        ValueError: If the month format is invalid.
        OSError: If a file operation or download fails.
    """
    parsed = pd.Period(month, freq="M")
    if str(parsed) != month:
        raise ValueError("Use a month in YYYY-MM format")
    data_dir.mkdir(parents=True, exist_ok=True)
    destination = data_dir / f"yellow_tripdata_{month}.parquet"
    if not destination.exists():
        partial = destination.with_suffix(".parquet.part")
        urlretrieve(
            f"https://d37ci6vzurychx.cloudfront.net/trip-data/{destination.name}",
            partial,
        )
        partial.replace(destination)
    return destination

source_path = download_taxi_month(TAXI_MONTH, DATA_DIR)
audit_columns = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
    "PULocationID", "DOLocationID", "passenger_count",
]
taxi_raw = pd.read_parquet(source_path, columns=audit_columns)
print("Original rows:", f"{len(taxi_raw):,}")
print("Columns:", taxi_raw.columns.tolist())
print(taxi_raw.head().to_string(index=False))


## 4. Preserve records and flag data issues
Duration is the difference between drop-off and pickup time. Durations at or below zero cannot be used as valid training labels. Long trips are flagged for investigation, not automatically deleted.

The prepared table retains every row. Flags are separate from prediction features. Completed-trip distance is used only for the data-quality audit at this stage, not as a pickup-time input.

Taxi timestamps are treated as local New York wall-clock times for this January audit. Before combining months or joining weather, verify the source time conventions and explicitly handle daylight-saving changes.


In [ ]:
def audit_taxi_records(frame: pd.DataFrame, month: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add audit fields without modifying the input or removing rows.

    Args:
        frame (pd.DataFrame): Selected source taxi columns.
        month (str): Expected pickup month in YYYY-MM format.
    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: Retained records and flag counts.
    """
    retained = frame.copy()
    pickup = pd.to_datetime(retained["tpep_pickup_datetime"], errors="coerce")
    dropoff = pd.to_datetime(retained["tpep_dropoff_datetime"], errors="coerce")
    duration = (dropoff - pickup).dt.total_seconds() / 60
    expected = pd.Period(month, freq="M")
    retained["duration_minutes"] = duration
    retained["flag_missing_timestamp"] = pickup.isna() | dropoff.isna()
    retained["flag_outside_pickup_month"] = ~(
        pickup.dt.year.eq(expected.year) & pickup.dt.month.eq(expected.month)
    )
    retained["flag_invalid_duration"] = ~np.isfinite(duration) | duration.le(0)
    retained["flag_duration_over_180_minutes"] = duration.gt(180)
    retained["flag_invalid_distance"] = (
        ~np.isfinite(retained["trip_distance"]) | retained["trip_distance"].lt(0)
    )
    retained["flag_zero_distance"] = retained["trip_distance"].eq(0)
    retained["flag_distance_over_100_miles"] = retained["trip_distance"].gt(100)
    retained["flag_missing_zone"] = retained[["PULocationID", "DOLocationID"]].isna().any(axis=1)
    flags = [column for column in retained if column.startswith("flag_")]
    report = retained[flags].sum().rename("flagged_rows").to_frame()
    report["percent_of_source"] = report["flagged_rows"] / len(retained) * 100
    assert len(retained) == len(frame)
    assert retained.index.equals(frame.index)
    return retained, report

taxi_audit, flag_report = audit_taxi_records(taxi_raw, TAXI_MONTH)
print(flag_report.to_string())
print("Rows removed:", len(taxi_raw) - len(taxi_audit))
print("Flag counts overlap: one row can have several flags.")
print(taxi_audit["duration_minutes"].describe(percentiles=[.01, .5, .95, .99]).to_string())
flag_report.to_csv(OUTPUT_DIR / "taxi_quality_flags.csv")


## 5. Weather audit before joining
Weather is not downloaded or joined yet. First select a source and document its station/location, variables, units, time zone, and observation periods. Hourly detail alone does not prove predictive value.

Questions to resolve:
- Does precipitation describe the previous hour, the following hour, or a daily total?
- Does a snow variable mean new snowfall, snow depth, or a condition code?
- When could this weather value have been known? Historical observations are not historical forecasts.
- Are New York local taxi times and weather times converted to a common time zone?
- How many distinct rainy, snowy, and dry hours occur in training and validation?
- How much weather is missing? Missing precipitation is not automatically dry weather.

Use a backward time match only after the weather timestamp represents a documented availability time. A row at 10:00 summarizing 10:00–11:00 must not be used for a 10:05 prediction. Never use forward or backward filling across the train/validation boundary without a justified rule.

Candidate sources: the Meteostat source in the assignment, or another documented weather source allowed by the assignment. No weather results are claimed at this stage.


## 6. Experiment design before training
1. Choose the modeling period after the audit. Preserve the original files and report training eligibility separately.
2. Use a chronological 80/20 training/validation division. Check that a training ride finishes before validation begins, not merely that its pickup is earlier. Document the resulting counts and any boundary exclusions.
3. If sampling is needed, fix the time boundary first. Keep sampling reproducible and report counts; model comparisons must use identical rows. Weather is shared across many trips, so also report the number of distinct weather hours.
4. Fit imputers, category vocabularies, scalers, and any route estimates on training data only. Zone IDs are categories, not continuous measurements.
5. Compare a simple training-only duration baseline with the three required TensorFlow architectures. Use 100 epochs and batch size 32 for every completed fit. Save training/validation loss curves for every fit and record failed runs honestly.
6. Choose an explicit experiment table covering both losses, all three optimizers, and multiple learning rates. A full Cartesian grid is not assumed to be an assignment requirement. Do not stop runs early just because a curve becomes less favorable; record numerical failures separately.
7. Rank runs using a common validation MAE in minutes; also report RMSE and R-squared. MSE loss and MAE loss have different units and cannot be ranked by raw loss values.
8. Compare weather and no-weather inputs on identical data. A comparison using validation is exploratory; a later untouched test period would support a stronger final claim.
9. Add the PyTorch DNN extra credit using matching architecture, split, preprocessing, loss, optimizer, epochs, and batch size. Seeds help reproducibility but do not make the frameworks numerically identical.

No model has been trained in this starter notebook. Findings, plots, model selection, predictions, and conclusions will be added after actual experiments.
